# Mouse pseudotime — variant comparison
Compares every mouse pseudotime that exists on disk (2026-09-01):

| tag | object | cells | what was dropped beyond v2 baseline* | pt columns |
|---|---|---|---|---|
| `baseline_v2` | `pseudotime_mouse_v2_induced_no_apop/…_rooted.h5ad` | 32,957 | — (manual root, iroot 6766) | `dpt_pseudotime` (rooted DPT), `palantir_pt` (Palantir from the same root; ρ=0.917 vs DPT) |
| `varA` | `results/pt_variants/varA_no_hiapo/` | 32,589 | 368 `decidual_mature` with apoptosis ≥ q25 of the apoptotic population (0.1077) | **none computed** — embedded only, never rooted |
| `varB` | `results/pt_variants/varB_no_efibouter/` | 31,891 | all 1,066 `efib_outer` | `palantir_pt` (root = `efib_proliferative` anchor, orientation-gated) |
| `varC` | `results/pt_variants/varC_no_both/` | 31,523 | both | `palantir_pt` |
| `varD` | `results/pt_variants/varD_keep_uninduced_no_efibouter/` | 40,217 | **keeps uninduced** (9,333 cells); drops `efib_outer` + annotated apoptotic; **no mature trim** (AUCell audit: score-flagged matures are gene-set-dependent) | **none yet** — root to be chosen in Part II; carries `apo_neighbor_frac` |

\* v2 baseline drops `decidual_apoptotic` and uninduced from the fib/dec deliverable, then retrains scVI.

**Caveat on `dpt_pseudotime` in the variants:** the variant objects carry a `dpt_pseudotime` column, but they were never rooted — it is **inherited from the input deliverable** (the v1 whole-lineage run), not computed on the variant embedding. It is loaded here only to prove that (section 5); it is not a variant result.

**What the earlier job assumed vs what it found:** the orientation gate ranked subtypes in the order
`efib_proliferative < efib_outer < efib_middle < efib_middle_induced < efib_subluminal < decidual_boundary < decidual_proliferative < decidual_mature`.
The data put `efib_middle_induced` *before* `efib_middle` in every variant. Section 4 looks at that directly.

Every headline number is recomputed here with `verify()`; a MISMATCH is a result, not a bug to hide.

In [ ]:
import pathlib, json, warnings
import numpy as np, pandas as pd, scanpy as sc, anndata as ad
import matplotlib.pyplot as plt, seaborn as sns
from scipy.stats import spearmanr, mannwhitneyu
warnings.filterwarnings("ignore")
sns.set_context("notebook"); plt.rcParams["figure.dpi"] = 110

X = pathlib.Path("/n/lab_storage/eddy_lab/users/nhilgert/xmens_final")
OBJ = {
 "baseline_v2": X/"pseudotime_mouse_v2_induced_no_apop/2026_08_28_mouse_fibdec_pseudotime_rooted.h5ad",
 "varA":        X/"results/pt_variants/varA_no_hiapo/varA_no_hiapo_embedded.h5ad",
 "varB":        X/"results/pt_variants/varB_no_efibouter/varB_no_efibouter_embedded.h5ad",
 "varC":        X/"results/pt_variants/varC_no_both/varC_no_both_embedded.h5ad",
 "varD":        X/"results/pt_variants/varD_keep_uninduced_no_efibouter/varD_keep_uninduced_no_efibouter_embedded.h5ad",  # uninduced KEPT, efib_outer + annotated apoptotic dropped, no mature trim
}
SUBORDER = ['efib_proliferative','efib_outer','efib_middle','efib_middle_induced',
            'efib_subluminal','decidual_boundary','decidual_proliferative','decidual_mature']
TPORDER  = ['uninduced','1_dpi_early','1_dpi_late','2_dpi','3_dpi','4_dpi_bleeding']
PAL = dict(zip(SUBORDER, sns.color_palette("viridis", len(SUBORDER))))

_checks = []
def verify(label, got, expected, tol=0.02, note=""):
    ok = abs(float(got) - float(expected)) <= tol if isinstance(expected,(int,float)) else got == expected
    _checks.append((label, got, expected, ok, note))
    print(f"{'PASS' if ok else 'MISMATCH':8s} {label}: got {got}  expected {expected}  {note}")
def check_summary():
    n_bad = sum(not c[3] for c in _checks)
    print(f"\n{len(_checks)} checks | {n_bad} MISMATCH")
for k, p in OBJ.items(): assert p.exists(), p
print(f"all {len(OBJ)} objects present")

## 1. Load `obs` (backed — no expression matrices in memory) and build one long table

In [ ]:
obs = {}
for k, p in OBJ.items():
    a = sc.read_h5ad(p, backed="r")
    o = a.obs.copy()
    o["umap_x"], o["umap_y"] = a.obsm["X_umap"][:,0], a.obsm["X_umap"][:,1]
    obs[k] = o
    print(f"{k:12s} n={a.n_obs:6,d}  pt cols: {[c for c in o.columns if 'pseudotime' in c or 'palantir' in c]}")
    a.file.close()

# methods = (variant, column) pairs that are REAL results of that variant
METHODS = [("baseline_v2","dpt_pseudotime"), ("baseline_v2","palantir_pt"),
           ("varB","palantir_pt"), ("varC","palantir_pt")]
rows = []
for var, col in METHODS:
    o = obs[var]
    rows.append(pd.DataFrame({"cell": o.index, "variant": var, "method": f"{var}:{col}",
        "pt": pd.to_numeric(o[col], errors="coerce").values,
        "subtype": o["cell_subtype"].astype(str).values,
        "timepoint": o["timepoint"].astype(str).values,
        "apoptosis": pd.to_numeric(o.get("apoptosis"), errors="coerce").values if "apoptosis" in o else np.nan}))
L = pd.concat(rows, ignore_index=True).dropna(subset=["pt"])
L["pt_rank"] = L.groupby("method").pt.rank(pct=True)   # percentile within each method/object: same orderings, cell-count spacing
L["subtype"]   = pd.Categorical(L.subtype,   [s for s in SUBORDER if s in L.subtype.unique()], ordered=True)
L["timepoint"] = pd.Categorical(L.timepoint, [t for t in TPORDER  if t in L.timepoint.unique()], ordered=True)
L.groupby("method", observed=True).pt.describe()[["count","min","25%","50%","75%","max"]].round(3)

## 2. What each variant removed — cell counts per subtype

In [ ]:
ct = pd.DataFrame({k: o["cell_subtype"].astype(str).value_counts() for k, o in obs.items()}).reindex(SUBORDER).fillna(0).astype(int)
ct.loc["TOTAL"] = ct.sum()
verify("varB removed all efib_outer", int(ct.loc["efib_outer","varB"]), 0)
verify("varA removed 368 matures", int(ct.loc["decidual_mature","baseline_v2"] - ct.loc["decidual_mature","varA"]), 368, tol=0)
verify("varC n", int(ct.loc["TOTAL","varC"]), 31523, tol=0)
ct

## 3. Pseudotime by subtype — violins, one panel per method
Medians annotated. Subtypes in the order the orientation gate *assumed*; watch `efib_middle_induced` vs `efib_middle`.

In [ ]:
methods = L.method.unique()
fig, axes = plt.subplots(len(methods), 2, figsize=(20, 3.0*len(methods)), sharex=True)
for i, m in enumerate(methods):
  for j, (ycol, ttl) in enumerate((("pt","raw"),("pt_rank","rank-normalised"))):
    ax = axes[i, j]
    d = L[L.method == m]
    sns.violinplot(data=d, x="subtype", y=ycol, hue="subtype", palette=PAL, legend=False,
                   cut=0, inner=None, density_norm="width", ax=ax)
    med = d.groupby("subtype", observed=True)[ycol].median()
    for k_, s in enumerate(d.subtype.cat.categories):
        if s in med: ax.plot([k_-0.35, k_+0.35], [med[s]]*2, c="k", lw=2); ax.text(k_, med[s]+0.02, f"{med[s]:.2f}", ha="center", fontsize=8)
    ax.set_title(f"{m} — {ttl}", loc="left", fontweight="bold", fontsize=9); ax.set_ylabel(ycol); ax.set_xlabel("")
for ax in axes[-1]: ax.tick_params(axis="x", rotation=30)
fig.tight_layout()

MED = L.groupby(["method","subtype"], observed=True).pt.median().unstack("subtype").round(3)
MED

### 3b. Subtype-rank ρ and IQR per method (reproduces the variant Palantir report)

In [ ]:
RANK = {s: i for i, s in enumerate(SUBORDER)}
summ = []
for m, d in L.groupby("method", observed=True):
    rr = spearmanr(d.pt, d.subtype.astype(str).map(RANK)).statistic
    iqr = np.subtract(*np.percentile(d.pt, [75, 25]))
    mono = bool(np.all(np.diff([MED.loc[m, s] for s in SUBORDER if s in MED.columns and np.isfinite(MED.loc[m, s])]) > 0))
    summ.append(dict(method=m, n=len(d), rho_subtype_rank=round(rr,4), IQR=round(iqr,4), medians_monotone_in_assumed_order=mono))
S = pd.DataFrame(summ).set_index("method")
verify("varB rho_subtype_rank", S.loc["varB:palantir_pt","rho_subtype_rank"], 0.8516, tol=0.005, note="variant_palantir_report.json")
verify("varC rho_subtype_rank", S.loc["varC:palantir_pt","rho_subtype_rank"], 0.8447, tol=0.005)
verify("varB IQR", S.loc["varB:palantir_pt","IQR"], 0.2119, tol=0.005)
verify("varC IQR", S.loc["varC:palantir_pt","IQR"], 0.1923, tol=0.005)
S

## 4. `efib_middle_induced` before `efib_middle` — is it real, and what is it?
Three views: (a) the two subtypes' pt distributions side by side with a Mann-Whitney test; (b) their timepoint composition — `_induced` cells exist only at induced timepoints by definition, so if pt tracks time this could be a composition effect; (c) pt of the two subtypes **within each timepoint**, which removes that confound.

In [ ]:
pair = L[L.subtype.isin(["efib_middle","efib_middle_induced"])].copy()
pair["subtype"] = pair.subtype.cat.remove_unused_categories()   # else seaborn wants palette keys for all 8
res = []
for m, d in pair.groupby("method", observed=True):
    a_ = d[d.subtype=="efib_middle"].pt; b_ = d[d.subtype=="efib_middle_induced"].pt
    u = mannwhitneyu(a_, b_, alternative="two-sided")
    res.append(dict(method=m, median_middle=a_.median(), median_middle_induced=b_.median(),
                    delta=b_.median()-a_.median(), n_middle=len(a_), n_induced=len(b_), mwu_p=u.pvalue))
R = pd.DataFrame(res).set_index("method").round(4)
display(R)

print("\n(b) timepoint composition of the two subtypes (baseline object):")
comp = pd.crosstab(obs["baseline_v2"]["cell_subtype"].astype(str), obs["baseline_v2"]["timepoint"].astype(str))
display(comp.loc[["efib_middle","efib_middle_induced"]].reindex(columns=[t for t in TPORDER if t in comp.columns]))

fig, axes = plt.subplots(1, len(methods), figsize=(4.2*len(methods), 3.8), sharey=True)
for ax, m in zip(axes, methods):
    d = pair[pair.method == m]
    sns.violinplot(data=d, x="timepoint", y="pt", hue="subtype", split=True, cut=0, inner="quart",
                   palette={"efib_middle": PAL["efib_middle"], "efib_middle_induced": PAL["efib_middle_induced"]}, ax=ax)
    ax.set_title(m, fontsize=9); ax.set_xlabel(""); ax.tick_params(axis="x", rotation=30)
    if ax is not axes[0]: ax.get_legend().remove()
fig.suptitle("(c) efib_middle vs efib_middle_induced, within timepoint", y=1.02); fig.tight_layout()

print("\nwithin-timepoint median difference (induced − middle), per method:")
wt = pair.groupby(["method","timepoint","subtype"], observed=True).pt.median().unstack("subtype")
wt["induced_minus_middle"] = wt["efib_middle_induced"] - wt["efib_middle"]
wt[["induced_minus_middle"]].unstack("timepoint").round(3)

**How to read section 4.** If `induced_minus_middle` is negative *within* timepoints too, the ordering is a property of the embedding, not of when the cells were sampled — i.e. the induced-middle state sits closer to the proliferative root than resting middle fibroblasts do, in every variant. That would argue the assumed rank order (`middle < middle_induced`) was wrong and `middle_induced` is an *earlier* response state, not a later one. If the within-timepoint deltas are ~0 or mixed, the pooled ordering is a composition artifact.

## 5. Method agreement on shared cells
Spearman between every pair of pseudotimes on the cells they share. Also proves the variants' `dpt_pseudotime` is inherited (compare to the baseline's DPT and to the deliverable's provenance).

In [ ]:
W = L.pivot_table(index="cell", columns="method", values="pt")
W_full = W.dropna()
print(f"cells shared by all four methods: {len(W_full):,}")
rho = W_full.corr(method="spearman").round(3)
display(rho)
verify("baseline palantir vs dpt rho", rho.loc["baseline_v2:palantir_pt","baseline_v2:dpt_pseudotime"], 0.917, tol=0.01, note="recorded when palantir was added to the rooted object")

# inherited dpt check: variant dpt column vs baseline rooted dpt on shared cells
for v in ("varB","varC"):
    o = obs[v]; b = obs["baseline_v2"]
    shared = o.index.intersection(b.index)
    r_ = spearmanr(pd.to_numeric(o.loc[shared,"dpt_pseudotime"]), pd.to_numeric(b.loc[shared,"dpt_pseudotime"])).statistic
    ident = np.allclose(pd.to_numeric(o.loc[shared,"dpt_pseudotime"]).values, pd.to_numeric(b.loc[shared,"dpt_pseudotime"]).values, equal_nan=True)
    print(f"{v}.dpt_pseudotime vs baseline rooted dpt: rho={r_:.3f}  byte-identical={ident}  -> "
          f"{'SAME source as baseline' if ident else 'a DIFFERENT dpt (inherited from the deliverable / v1 run) — not a variant result'}")

g = sns.pairplot(W_full.sample(min(8000, len(W_full)), random_state=0), corner=True,
                 plot_kws=dict(s=3, alpha=0.3, color="k"), diag_kind="hist")
g.figure.suptitle("pairwise pseudotimes on shared cells (8k sample)", y=1.02);

## 6. Pseudotime by timepoint

In [ ]:
fig, axes = plt.subplots(len(methods), 2, figsize=(11, 3.0*len(methods)), sharex=True)
tpal = dict(zip(TPORDER, sns.color_palette("rocket_r", len(TPORDER))))
for i, m in enumerate(methods):
  for j, (ycol, ttl) in enumerate((("pt","raw"),("pt_rank","rank-normalised"))):
    ax = axes[i, j]; d = L[L.method == m]
    sns.violinplot(data=d, x="timepoint", y=ycol, hue="timepoint", palette=tpal, legend=False, cut=0, inner=None, density_norm="width", ax=ax)
    med = d.groupby("timepoint", observed=True)[ycol].median()
    for k_, t in enumerate(d.timepoint.cat.categories):
        if t in med: ax.plot([k_-0.35, k_+0.35], [med[t]]*2, c="k", lw=2); ax.text(k_, med[t]+0.02, f"{med[t]:.2f}", ha="center", fontsize=8)
    rho = spearmanr(d[ycol], d.timepoint.cat.codes).statistic
    ax.set_title(f"{m} — {ttl}   ρ(pt, day)={rho:+.2f}", loc="left", fontsize=9, fontweight="bold"); ax.set_xlabel(""); ax.set_ylabel(ycol)
for ax in axes[-1]: ax.tick_params(axis="x", rotation=25)
fig.tight_layout()
# NOTE: every induced section holds BOTH compartments, so pooled violins are bimodal (fibroblast lobe + decidual lobe) and
# rho(pt, day) ~0.4 is expected. The orientation-relevant check is the decidual compartment's median by day:
L["comp"] = np.where(L.subtype.astype(str).str.startswith("decidual"), "decidual", "fibroblast")
L.groupby(["method","comp","timepoint"], observed=True).pt_rank.median().unstack("timepoint").round(2)

## 7. Embeddings — each variant's own UMAP, coloured by its pseudotime and by subtype

In [ ]:
show = [("baseline_v2","dpt_pseudotime"), ("baseline_v2","palantir_pt"), ("varB","palantir_pt"), ("varC","palantir_pt"), ("varA", None)]
fig, axes = plt.subplots(2, len(show), figsize=(4*len(show), 7.5))
for j, (v, col) in enumerate(show):
    o = obs[v]
    if col:
        sc_ = axes[0,j].scatter(o.umap_x, o.umap_y, c=pd.to_numeric(o[col]), s=1.5, cmap="viridis", rasterized=True)
        plt.colorbar(sc_, ax=axes[0,j], fraction=0.04); axes[0,j].set_title(f"{v}: {col}", fontsize=9)
    else:
        axes[0,j].text(0.5,0.5,"varA: embedded,\nno pseudotime computed", ha="center", va="center", transform=axes[0,j].transAxes); axes[0,j].set_title("varA", fontsize=9)
    for s in SUBORDER:
        mm = o["cell_subtype"].astype(str) == s
        if mm.any(): axes[1,j].scatter(o.umap_x[mm], o.umap_y[mm], s=1.5, c=[PAL[s]], label=s, rasterized=True)
    axes[1,j].set_title(f"{v}: subtypes (n={len(o):,})", fontsize=9)
    for ax in axes[:,j]: ax.set_xticks([]); ax.set_yticks([])
axes[1,0].legend(markerscale=6, fontsize=7, loc="best")
fig.tight_layout()

## 8. Apoptosis score along pseudotime
The motivation for varA/varC was matures drifting toward an apoptotic signature. Does high-apoptosis-score sit at the pt extreme in each method?

In [ ]:
A = L.dropna(subset=["apoptosis"]).copy()
A["pt_bin"] = pd.cut(A.pt, np.linspace(0, 1, 21), include_lowest=True)
fig, axes = plt.subplots(1, len(methods), figsize=(4.2*len(methods), 3.6), sharey=True)
for ax, m in zip(axes, methods):
    d = A[A.method == m]
    prof = d.groupby("pt_bin", observed=True).apoptosis.agg(["median", lambda v: v.quantile(.9)])
    prof.columns = ["median","q90"]; xs = [iv.mid for iv in prof.index]
    ax.plot(xs, prof["median"], c="k", label="median"); ax.plot(xs, prof["q90"], c="r", ls="--", label="q90")
    ax.axhline(0.1077, c="gray", lw=0.8, ls=":", label="varA/varC hi-apo threshold")
    ax.set_title(m, fontsize=9); ax.set_xlabel("pseudotime (20 bins)")
axes[0].set_ylabel("apoptosis score"); axes[0].legend(fontsize=7)
fig.tight_layout()
A.groupby("method", observed=True).apply(lambda d: spearmanr(d.pt, d.apoptosis).statistic).round(3).rename("spearman(pt, apoptosis)")

## 9. Summary

In [ ]:
check_summary()
out = S.join(R[["delta","mwu_p"]].rename(columns={"delta":"middle_induced_minus_middle_median","mwu_p":"middle_pair_mwu_p"}))
out

### Where this leaves the decision (fill in after running)
- **Method:** on the same cells Palantir and rooted DPT agree at ρ≈0.92 (section 5); the method switch alone changes little.
- **Subsetting:** dropping `efib_outer` (varB) is what opens the dynamic range (IQR 0.21 vs baseline); also dropping hi-apo matures (varC) adds nothing over B.
- **The `middle_induced` question:** decided by section 4(c) — if the within-timepoint deltas are consistently negative, the assumed rank order was wrong and the ρ_subtype_rank values above are *understated*, not overstated.
- **Nothing downstream uses any variant yet** — MI, transfers and the deliverable all use `baseline_v2:dpt_pseudotime`. Adopting varB means re-running `mouse_dpt` MI and the pt-dependent transfers on 31,891 cells; isodepth-based results are unaffected.

---
# Part II — make the root explicit, choose it, recompute

**Why:** section 3 shows the root is off in both generations. Baseline (iroot 6766) starts in `efib_outer`; varB/varC used the *single* `efib_proliferative` cell farthest from the rest (`anchor_far`), an outlier — the cells at pt≈0 are actually `efib_subluminal`, and `efib_proliferative`'s median lands *above* `efib_middle`. The whole fibroblast side is squeezed into 0.2–0.3 while `decidual_mature` claims 0.2→1.0.

**Convention:** scanpy's `X_diffmap[:, 0]` is the trivial steady-state component; **DC1 = column 1, DC2 = column 2**, etc. Plots below are labelled by column index.

Run Part II on a **compute node** (Palantir on ~32k cells × 4 objects is minutes each, not seconds). Nothing here writes to any h5ad — results go to `results/pt_variants/manual_roots/*.csv` until you decide to adopt one.

## 10. Root inspection — diffusion components with the CURRENT root marked

In [ ]:
import h5py, re
try:
    from anndata.io import read_elem
except ImportError:
    from anndata.experimental import read_elem

DM, ROOT_NOW = {}, {}
for k, p in OBJ.items():
    with h5py.File(p, "r") as f:
        DM[k] = read_elem(f["obsm/X_diffmap"])
        uns_keys = list(f["uns"].keys())
        iroot = int(read_elem(f["uns/iroot"])) if "iroot" in uns_keys else None
    o = obs[k]
    if iroot is not None:
        ROOT_NOW[k] = dict(barcode=o.index[iroot], idx=iroot, source="uns['iroot']")
    elif "palantir_root_desc" in o.columns:
        desc = str(o["palantir_root_desc"].iloc[0]); m = re.search(r"\((\S+)\)", desc)
        bc = m.group(1) if m else None
        ROOT_NOW[k] = dict(barcode=bc, idx=(o.index.get_loc(bc) if bc in o.index else None), source=f"palantir_root_desc: {desc}")
    else:
        ROOT_NOW[k] = dict(barcode=None, idx=None, source="none (never rooted)")
    r = ROOT_NOW[k]
    print(f"{k:12s} DCs={DM[k].shape[1]}  root={r['barcode']}  subtype={o['cell_subtype'].iloc[r['idx']] if r['idx'] is not None else '-'}"
          f"  tp={o['timepoint'].iloc[r['idx']] if r['idx'] is not None else '-'}  [{r['source']}]")

In [ ]:
PAIRS = [(1,2), (2,3), (1,3), (2,4)]   # (col_x, col_y) into X_diffmap; DC2 vs DC3 is the one you asked for

def dc_panels(k, color="subtype"):
    o, E, r = obs[k], DM[k], ROOT_NOW[k]
    fig, axes = plt.subplots(1, len(PAIRS), figsize=(4.6*len(PAIRS), 4.2))
    for ax, (i, j) in zip(axes, PAIRS):
        if color == "subtype":
            for s in SUBORDER:
                mm = (o["cell_subtype"].astype(str) == s).values
                if mm.any(): ax.scatter(E[mm, i], E[mm, j], s=2, c=[PAL[s]], label=s, rasterized=True)
        else:
            tps = [t for t in TPORDER if t in set(o["timepoint"].astype(str))]
            for t, col in zip(tps, sns.color_palette("rocket", len(tps))):
                mm = (o["timepoint"].astype(str) == t).values
                ax.scatter(E[mm, i], E[mm, j], s=2, c=[col], label=t, rasterized=True)
        if r["idx"] is not None:
            ax.scatter(E[r["idx"], i], E[r["idx"], j], s=260, marker="*", c="red", edgecolor="k", lw=1, zorder=5, label="CURRENT root")
        ax.set_xlabel(f"DC{i}"); ax.set_ylabel(f"DC{j}"); ax.set_xticks([]); ax.set_yticks([])
    axes[0].legend(markerscale=5, fontsize=7, loc="best")
    fig.suptitle(f"{k} — diffusion components, coloured by {color}; red star = current root", y=1.02)
    fig.tight_layout()

for k in OBJ:
    dc_panels(k, "subtype")
    dc_panels(k, "timepoint")

### 10b. Which subtype sits at each DC extreme? (helps pick a root by coordinate rather than by eye alone)

In [ ]:
rows = []
for k in OBJ:
    o, E = obs[k], DM[k]
    sub = o["cell_subtype"].astype(str).values
    for c in range(1, min(6, E.shape[1])):
        lo, hi = np.argsort(E[:, c])[:200], np.argsort(E[:, c])[-200:]   # 200 most extreme cells each side
        rows.append(dict(object=k, DC=f"DC{c}",
                         low_end=pd.Series(sub[lo]).value_counts().index[0],  low_frac=round(pd.Series(sub[lo]).value_counts(normalize=True).iloc[0],2),
                         high_end=pd.Series(sub[hi]).value_counts().index[0], high_frac=round(pd.Series(sub[hi]).value_counts(normalize=True).iloc[0],2)))
pd.DataFrame(rows).set_index(["object","DC"])

## 11. Choose the root — EDIT THIS CELL
Two ways to specify, per object:
- `dict(barcode="AAAC...")` — an explicit cell you read off the plots, or
- `dict(subtype="efib_proliferative", dc=2, side="low", pct=2)` — the cell at the **2nd percentile** of DC2 *within that subtype* (a robust extreme; `pct=0` would reproduce the outlier-anchor behaviour that misfired before). Use `side="high"` for the other end.

Defaults below are a starting proposal only — put the root in the proliferative pool's DC-extreme, not in an isolated cell. Look at section 10 first and change them.

In [ ]:
ROOT_SPEC = {
    "baseline_v2": dict(dc=1, side="low",  pct=0),
    "varA":        dict(dc=1, side="low",  pct=0),
    "varB":        dict(dc=1, side="low",  pct=0),
    "varC":        dict(dc=1, side="high", pct=0),
    # varD has the true start state available: root INSIDE the uninduced resting fibroblasts.
    # Check the DC plots in section 10 first and set dc/side to the uninduced pole; pct=2 avoids a single-outlier root.
    "varD":        dict(timepoint="uninduced", subtype="efib_middle", dc=1, side="low", pct=2),
}
# keys: barcode=... (explicit)  |  dc, side ("low"/"high"), pct (0 = single most extreme cell; >0 = robust percentile)
#       optional filters: subtype=..., timepoint=...  (the extreme is searched only among cells passing all filters)

def resolve_root(k, spec):
    o, E = obs[k], DM[k]
    if "barcode" in spec:
        bc = spec["barcode"]; assert bc in o.index, f"{bc} not in {k}"
        return bc, o.index.get_loc(bc)
    mm = np.ones(len(o), bool)
    if "subtype" in spec:   mm &= (o["cell_subtype"].astype(str) == spec["subtype"]).values
    if "timepoint" in spec: mm &= (o["timepoint"].astype(str) == spec["timepoint"]).values
    assert mm.any(), f"no cells in {k} match {spec}"
    vals = E[mm, spec["dc"]]
    if spec.get("pct", 0) == 0:
        j = int(np.argmin(vals)) if spec["side"] == "low" else int(np.argmax(vals))
    else:
        q = np.percentile(vals, spec["pct"] if spec["side"] == "low" else 100 - spec["pct"])
        j = int(np.argmin(np.abs(vals - q)))
    idx = np.flatnonzero(mm)[j]
    return o.index[idx], idx

ROOT_NEW = {}
for k, spec in ROOT_SPEC.items():
    if k not in OBJ: continue
    bc, idx = resolve_root(k, spec)
    ROOT_NEW[k] = dict(barcode=bc, idx=int(idx), spec=spec)
    o = obs[k]
    filt = ", ".join(f"{f}={spec[f]}" for f in ("subtype","timepoint") if f in spec) or "all cells"
    print(f"{k:12s} NEW root {bc}  subtype={o['cell_subtype'].iloc[idx]}  tp={o['timepoint'].iloc[idx]}  "
          f"DC{spec['dc']}={DM[k][idx, spec['dc']]:.4f}  ({spec['side']} extreme, pct={spec.get('pct',0)}, {filt})")

# old vs new root on DC1 x DC2 for every object
fig, axes = plt.subplots(1, len(OBJ), figsize=(4.8*len(OBJ), 4.4))
for ax, k in zip(np.atleast_1d(axes), OBJ):
    o, E = obs[k], DM[k]
    for s in SUBORDER:
        mm = (o["cell_subtype"].astype(str) == s).values
        if mm.any(): ax.scatter(E[mm,1], E[mm,2], s=2, c=[PAL[s]], rasterized=True)
    if ROOT_NOW[k]["idx"] is not None:
        ax.scatter(E[ROOT_NOW[k]["idx"],1], E[ROOT_NOW[k]["idx"],2], s=220, marker="*", c="red", edgecolor="k", label="current")
    ax.scatter(E[ROOT_NEW[k]["idx"],1], E[ROOT_NEW[k]["idx"],2], s=220, marker="P", c="cyan", edgecolor="k", label="NEW")
    ax.set_title(k); ax.set_xlabel("DC1"); ax.set_ylabel("DC2"); ax.set_xticks([]); ax.set_yticks([]); ax.legend(fontsize=8)
fig.tight_layout()

## 12. Recompute DPT **and** Palantir from the chosen root, per object
- **DPT**: same diffusion map already on the object (`X_diffmap`, `diffmap_evals`, neighbors graph) — only `iroot` changes.
- **Palantir**: diffusion maps on `X_scVI` → multiscale space → `run_palantir(ms, start_cell, num_waypoints=1200)`, same call the earlier job used.
- **Orientation gate**: Spearman with the subtype rank AND with timepoint order must both be > 0. If not, the result is marked **REFUSED** and kept as-is for inspection — never silently negated (standing rule).
- Output: `results/pt_variants/manual_roots/<object>_manual_pt.csv` (cell, dpt_manual, palantir_manual, root, gates). No h5ad is modified.

In [ ]:
import palantir, os
OUT = X/"results/pt_variants/manual_roots"; OUT.mkdir(parents=True, exist_ok=True)
TPRANK = {t: i for i, t in enumerate(TPORDER)}

def light_adata(p):
    """obs + obsm + neighbors graph + diffmap eigenvalues, no expression matrix."""
    with h5py.File(p, "r") as f:
        a = ad.AnnData(obs=read_elem(f["obs"]))
        for key in ("X_diffmap", "X_scVI", "X_umap"): a.obsm[key] = read_elem(f[f"obsm/{key}"])
        for key in ("connectivities", "distances"):
            if key in f["obsp"]: a.obsp[key] = read_elem(f[f"obsp/{key}"])
        for key in ("neighbors", "diffmap_evals"):
            if key in f["uns"]: a.uns[key] = read_elem(f[f"uns/{key}"])
    return a

def gates(pt, o):
    sub_r = o["cell_subtype"].astype(str).map(RANK).to_numpy(float)
    tp_r  = o["timepoint"].astype(str).map(TPRANK).to_numpy(float)
    ok = np.isfinite(pt)
    r_sub = spearmanr(pt[ok], sub_r[ok]).statistic
    m2 = ok & np.isfinite(tp_r)
    r_tp = spearmanr(pt[m2], tp_r[m2]).statistic if m2.sum() > 10 else np.nan
    verdict = "OK" if (r_sub > 0 and (np.isnan(r_tp) or r_tp > 0)) else "REFUSED (orientation gate failed — not flipped)"
    return dict(rho_subtype_rank=round(float(r_sub),4), rho_timepoint=round(float(r_tp),4) if np.isfinite(r_tp) else None, verdict=verdict)

MANUAL = {}
for k, p in OBJ.items():
    print(f"\n=== {k} ===", flush=True)
    a = light_adata(p); root = ROOT_NEW[k]
    res = pd.DataFrame(index=a.obs_names); res.index.name = "cell"
    # --- DPT on the existing diffmap
    if "diffmap_evals" in a.uns and "neighbors" in a.uns:
        a.uns["iroot"] = root["idx"]; sc.tl.dpt(a, n_dcs=min(10, a.obsm["X_diffmap"].shape[1]))
        res["dpt_manual"] = a.obs["dpt_pseudotime"].to_numpy(float)
        g = gates(res["dpt_manual"].to_numpy(), a.obs); print("  DPT     ", g, flush=True); g_dpt = g
    else:
        print("  DPT: object lacks diffmap_evals/neighbors — skipped"); g_dpt = None
    # --- Palantir on X_scVI
    df = pd.DataFrame(a.obsm["X_scVI"], index=a.obs_names)
    dm = palantir.utils.run_diffusion_maps(df, n_components=10, knn=30)
    ms = palantir.utils.determine_multiscale_space(dm)
    pr = palantir.core.run_palantir(ms, root["barcode"], num_waypoints=1200, n_jobs=8)
    res["palantir_manual"] = pr.pseudotime.reindex(a.obs_names).to_numpy(float)
    g = gates(res["palantir_manual"].to_numpy(), a.obs); print("  Palantir", g, flush=True); g_pal = g
    res["root_barcode"] = root["barcode"]; res["root_spec"] = json.dumps(root["spec"])
    res.to_csv(OUT/f"{k}_manual_pt.csv")
    json.dump(dict(root=root, dpt=g_dpt, palantir=g_pal), open(OUT/f"{k}_manual_root_report.json","w"), indent=1, default=str)
    MANUAL[k] = res
    del a
print("\nwritten to", OUT)

## 13. Before vs after — violins for the recomputed pseudotimes (raw and rank-normalised)
Rank-normalisation (pt → percentile among all cells) keeps every ordering but stops the `decidual_mature` tail from owning half the axis, so the fibroblast side becomes readable.

In [ ]:
rows = []
for k, res in MANUAL.items():
    o = obs[k]
    for col in ("dpt_manual", "palantir_manual"):
        if col not in res: continue
        pt = res[col].to_numpy(float)
        rows.append(pd.DataFrame({"cell": res.index, "method": f"{k}:{col}", "pt": pt,
            "pt_rank": pd.Series(pt).rank(pct=True).values,
            "subtype": o["cell_subtype"].astype(str).values, "timepoint": o["timepoint"].astype(str).values}))
M = pd.concat(rows, ignore_index=True).dropna(subset=["pt"])
M["subtype"] = pd.Categorical(M.subtype, [s for s in SUBORDER if s in M.subtype.unique()], ordered=True)
M["timepoint"] = pd.Categorical(M.timepoint, [t for t in TPORDER if t in M.timepoint.unique()], ordered=True)

for ycol, title in (("pt", "raw pseudotime"), ("pt_rank", "rank-normalised pseudotime (percentile)")):
    ms_ = M.method.unique()
    fig, axes = plt.subplots(len(ms_), 1, figsize=(11, 2.9*len(ms_)), sharex=True)
    for ax, m in zip(np.atleast_1d(axes), ms_):
        d = M[M.method == m]
        sns.violinplot(data=d, x="subtype", y=ycol, hue="subtype", palette=PAL, legend=False, cut=0, inner=None, density_norm="width", ax=ax)
        med = d.groupby("subtype", observed=True)[ycol].median()
        for i, s in enumerate(d.subtype.cat.categories):
            if s in med: ax.plot([i-0.35, i+0.35], [med[s]]*2, c="k", lw=2); ax.text(i, med[s]+0.02, f"{med[s]:.2f}", ha="center", fontsize=8)
        ax.set_title(m, loc="left", fontweight="bold"); ax.set_xlabel(""); ax.set_ylabel(ycol)
    np.atleast_1d(axes)[-1].tick_params(axis="x", rotation=30)
    fig.suptitle(f"manual roots — {title}", y=1.01); fig.tight_layout()

# side-by-side summary: old vs new
new_summ = []
for m, d in M.groupby("method", observed=True):
    new_summ.append(dict(method=m, rho_subtype_rank=round(spearmanr(d.pt, d.subtype.astype(str).map(RANK)).statistic,4),
                         IQR=round(np.subtract(*np.percentile(d.pt,[75,25])),4),
                         median_efib_prolif=round(d[d.subtype=="efib_proliferative"].pt.median(),3),
                         median_efib_middle=round(d[d.subtype=="efib_middle"].pt.median(),3),
                         median_efib_middle_induced=round(d[d.subtype=="efib_middle_induced"].pt.median(),3),
                         median_efib_subluminal=round(d[d.subtype=="efib_subluminal"].pt.median(),3),
                         median_dec_mature=round(d[d.subtype=="decidual_mature"].pt.median(),3)))
pd.concat([S.assign(roots="CURRENT")[["roots","rho_subtype_rank","IQR"]],
           pd.DataFrame(new_summ).set_index("method").assign(roots="MANUAL")]).sort_index()

# --- by TIMEPOINT, rank-normalised (varD adds the uninduced column at the left: should be a single fibroblast lobe near 0 if the root is right)
ms_ = M.method.unique()
fig, axes = plt.subplots(len(ms_), 1, figsize=(9, 2.7*len(ms_)), sharex=True)
for ax, m in zip(np.atleast_1d(axes), ms_):
    d = M[M.method == m]
    sns.violinplot(data=d, x="timepoint", y="pt_rank", hue="timepoint", palette=tpal, legend=False, cut=0, inner=None, density_norm="width", ax=ax)
    med = d.groupby("timepoint", observed=True).pt_rank.median()
    for k_, t in enumerate(d.timepoint.cat.categories):
        if t in med: ax.plot([k_-0.35, k_+0.35], [med[t]]*2, c="k", lw=2); ax.text(k_, med[t]+0.02, f"{med[t]:.2f}", ha="center", fontsize=8)
    ax.set_title(f"{m} — rank-normalised, by timepoint", loc="left", fontsize=9, fontweight="bold"); ax.set_xlabel(""); ax.set_ylabel("pt_rank")
np.atleast_1d(axes)[-1].tick_params(axis="x", rotation=25); fig.tight_layout()

### Reading Part II
- **A root is right when** `efib_proliferative` is the lowest-median subtype *and* its violin sits at pt≈0 — not merely "the root cell is a proliferative cell." The `anchor_far` root satisfied the second and failed the first.
- Compare `rho_subtype_rank` CURRENT vs MANUAL in the last table; remember the assumed order puts `efib_middle` before `efib_middle_induced`, which section 4 may have shown is backwards — a lower ρ can be the data disagreeing with the *order*, not a worse root.
- `decidual_mature`'s range persists under any root: it is the terminal population and the axis is scaled to its farthest cell. Use the rank-normalised panel for reading the fibroblast side; use section 8 to see whether the tail is the apoptosis-drifting cells (which would argue for varA/varC-style trimming, or for reporting pt clipped at the 99th percentile).
- Nothing has been written to any object. To adopt a manual pseudotime: pick the object+method, and the CSV in `results/pt_variants/manual_roots/` is the join table — that step is a separate, approved write.

## 14. Apoptosis scoring audit — why matures are NOT score-trimmed in varD
`results/pt_variants/apoptosis_aucell_v3_pyscenic/apoptosis_aucell_pyscenic.csv`: every fib/dec cell of the deliverable scored with **your** `score_aucell` convention (pyscenic `aucell`, CP10k from counts, `derive_auc_threshold[0.01]` ≈ 444 ranks — identical to the zone modules' `uns['aucell']`) on four apoptosis sets — MSigDB mouse `HALLMARK_APOPTOSIS` (161), `REACTOME_APOPTOSIS` (103), `REACTOME_APOPTOTIC_EXECUTION_PHASE` (49), moscot mouse (186) — with the 11 zone-module AUCs (`auc_PDZ`, `auc_SDZ`, …) copied alongside, same scale.

Questions this answers: (a) does any set separate a discrete apoptotic-like subpopulation inside `decidual_mature`? (b) do different sets flag the same matures? (c) what are the apoptosis sets actually reading in this tissue?

In [ ]:
from sklearn.metrics import roc_auc_score
from sklearn.mixture import GaussianMixture
AP = pd.read_csv(X/"results/pt_variants/apoptosis_aucell_v3_pyscenic/apoptosis_aucell_pyscenic.csv", index_col=0)
APSETS = ["hallmark_apoptosis","reactome_apoptosis","reactome_execution_phase","moscot_mouse"]
ZONES  = [c for c in AP.columns if c.startswith("auc_") and c.replace("auc_","") not in APSETS]
mat, apo = AP[AP.cell_subtype=="decidual_mature"], AP[AP.cell_subtype=="decidual_apoptotic"]
print(f"matures {len(mat):,} | annotated apoptotic {len(apo):,}")

# (a) distributions: mature vs apoptotic vs fibroblast, per set
fig, axes = plt.subplots(1, len(APSETS), figsize=(4.4*len(APSETS), 3.6))
for ax, s in zip(axes, APSETS):
    for lab_, grp, col in (("fibroblast", AP[AP.cell_subtype.str.startswith("efib")], "C0"), ("decidual_mature", mat, "C2"), ("decidual_apoptotic", apo, "C3")):
        sns.kdeplot(grp[f"auc_{s}"], ax=ax, label=lab_, color=col, fill=True, alpha=0.25, cut=0)
    ax.axvline(apo[f"auc_{s}"].quantile(.25), ls=":", c="C3", lw=1); ax.set_title(s, fontsize=9); ax.set_xlabel("AUCell")
axes[0].legend(fontsize=8); fig.tight_layout()

rows = []
for s in APSETS:
    m, a_ = mat[f"auc_{s}"].values, apo[f"auc_{s}"].values
    g1 = GaussianMixture(1, random_state=0).fit(m[:,None]); g2 = GaussianMixture(2, random_state=0).fit(m[:,None])
    rows.append(dict(set=s, AUROC_apo_vs_mature=round(roc_auc_score(np.r_[np.zeros(len(m)),np.ones(len(a_))], np.r_[m,a_]),3),
                     dBIC_2v1=round(g1.bic(m[:,None])-g2.bic(m[:,None]),1), hi_mode_weight=round(g2.weights_[np.argmax(g2.means_.ravel())],2),
                     n_top10pct=int((m>=np.percentile(m,90)).sum()), n_ge_apo_q25=int((m>=np.percentile(a_,25)).sum()), n_ge_apo_q50=int((m>=np.percentile(a_,50)).sum())))
SUMM = pd.DataFrame(rows).set_index("set")
verify("hallmark AUROC apoptotic vs mature", SUMM.loc["hallmark_apoptosis","AUROC_apo_vs_mature"], 0.875, tol=0.01)
SUMM

In [ ]:
# (b) do the sets flag the SAME matures?  top-10% per set, pairwise Jaccard + consensus
top = {s: set(mat.index[mat[f"auc_{s}"] >= mat[f"auc_{s}"].quantile(.9)]) for s in APSETS[:3]}
J = pd.DataFrame([[len(top[a]&top[b])/len(top[a]|top[b]) for b in APSETS[:3]] for a in APSETS[:3]], index=APSETS[:3], columns=APSETS[:3]).round(2)
n_sets = pd.Series(0, index=mat.index)
for s in APSETS[:3]: n_sets[list(top[s])] += 1
print("matures in top-10% of N standard sets:", n_sets.value_counts().sort_index().to_dict())
verify("hallmark∩execution top-10% Jaccard (<0.15 = set-dependent)", J.loc["hallmark_apoptosis","reactome_execution_phase"] < 0.15, True)
display(J)
# legacy rule for reference: old moscot score_genes >= 0.1077 flagged 368 matures (varA/varC); how many of those are consensus?
cons = set(n_sets.index[n_sets >= 2]); print(f"consensus (>=2 of 3 sets): {len(cons)} matures ({100*len(cons)/len(mat):.1f}%)")

In [ ]:
# (c) what are the apoptosis sets reading? correlate against the zone modules (same AUCell currency)
corr_all = AP[[f"auc_{s}" for s in APSETS[:3]] + ZONES].corr("spearman").loc[[f"auc_{s}" for s in APSETS[:3]], ZONES]
corr_mat = mat[[f"auc_{s}" for s in APSETS[:3]] + ZONES].corr("spearman").loc[[f"auc_{s}" for s in APSETS[:3]], ZONES]
fig, axes = plt.subplots(1, 2, figsize=(15, 3.2))
for ax, cc, ttl in zip(axes, (corr_all, corr_mat), ("all fib/dec cells", "decidual_mature only")):
    sns.heatmap(cc.rename(index=lambda i: i.replace("auc_",""), columns=lambda c: c.replace("auc_","")), cmap="RdBu_r", center=0, vmin=-0.7, vmax=0.7, annot=True, fmt=".2f", ax=ax, cbar=ax is axes[1])
    ax.set_title(f"Spearman(apoptosis AUC, zone-module AUC) — {ttl}", fontsize=9)
fig.tight_layout()
verify("execution-phase tracks SDZ across cells (rho>0.5)", corr_all.loc["auc_reactome_execution_phase","auc_SDZ"] > 0.5, True,
       note="the 'apoptotic execution' genes (Hmgb1/2, Lmna, Gsn, histones) are part of the SDZ differentiation program here")

**Reading §14.** The annotated `decidual_apoptotic` cluster separates cleanly from matures (Hallmark AUROC ≈ 0.88) — dropping it is justified. But inside `decidual_mature` there is no bimodality under any set, the top-10% flagged by different standard sets barely overlap (Jaccard ≈ 0.06), and the Reactome apoptosis scores track **SDZ identity** (ρ 0.5–0.6) rather than death — their genes are part of the decidual differentiation program in this tissue. A score threshold on matures therefore removes a set of cells defined by the gene list, not by biology. varD keeps all matures; the mature tail is handled by rank-normalisation (§13). If a trim is ever wanted, the ≥2-of-3 consensus (~7%) or the graph-neighbourhood flag (`apo_neighbor_frac` on varD; 8 matures with majority-apoptotic neighbours) are the only versions to report, as sensitivity checks.